In [2]:
import os
import json
import re
from dotenv import load_dotenv
from langchain_groq import ChatGroq

from langchain.tools import tool
import requests
from bs4 import BeautifulSoup

from langchain.agents import create_agent

load_dotenv()

model = ChatGroq(model ="llama-3.3-70b-versatile",
                 api_key=os.getenv("GROQ_API_KEY"),
                 temperature = 0)

In [3]:
@tool
def extract_json(text):

    """
    Extracts and returns the first valid JSON object from a text string.
    Returns a Python dictionary if successful, otherwise None.
    """
    
    match = re.search(r'\{.*\}', text, re.DOTALL)
    return match.group(0) if match else text


def country_pref(value):
    if not value:
        return ""
    value = value.lower()
    if "outside" in value or "foreign" in value or "international" in value or "non" in value or "not" in value:
        return "foreign"
    if "usa" in value or "us" in value or "america" in value:
        return "us"
    return ""


def is_invalid_output(data):
    return data.get("invalid") == True

In [4]:
@tool
def build_url(filters):

    """
    Build a Planefax aircraft search URL from filter dictionary.
    """
    
    base_url = "https://planefax.com/aircraft/for-sale/"
    params = []

    if filters.get("cat"):
        for c in filters["cat"]:
            params.append(f"cat={c}")

    if filters.get("manufacturer"):
        for m in filters["manufacturer"]:
            params.append(f"manufacturer={m}")

    params.append(f"year__gte={filters.get('year__gte') or ''}")
    params.append(f"year__lte={filters.get('year__lte') or ''}")
    params.append(f"min_price={filters.get('min_price') or ''}")
    params.append(f"max_price={filters.get('max_price') or ''}")

    if filters.get("state"):
        for s in filters["state"]:
            params.append(f"state={s}")

    country = country_pref(filters.get("country_pref"))
    params.append(f"country_pref={country}")

    params.append(f"min_date={filters.get('min_date') or ''}")
    params.append(f"max_date={filters.get('max_date') or ''}")

    final_url = base_url + "?" + "&".join(params)

    return final_url

In [5]:
@tool
def scrape_aircrafts(url):

    """
    Scrape aircraft listings from a Planefax URL.
    """
    
    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }

        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, "html.parser")

        aircrafts = []

        listings = soup.find_all("div", class_="listing-box")

        for item in listings[:5]:

            title_tag = item.find("h4", class_="listing-box-title")
            name = title_tag.text.strip() if title_tag else "N/A"

            price_tag = item.find("p", class_="listing-box-price")
            price = price_tag.text.strip() if price_tag else "N/A"

            location_tag = item.find("div", class_="listing-box-meta")
            location = location_tag.get_text(" ", strip=True) if location_tag else "N/A"

            aircrafts.append({
                "name": name,
                "price": price,
                "location": location
            })

        return json.dumps(aircrafts)

    except Exception as e:
        return json.dumps({"error": str(e)})


In [6]:
system_prompt = """
You are an aircraft search assistant.

Steps:
1. Extract filters from the user query
   Return JSON with this structure:
{
"cat": [],
"manufacturer": [],
"year__gte": null,
"year__lte": null,
"min_price": null,
"max_price": null,
"state": [],
"country_pref": null,
"min_date": null,
"max_date": null
}

DO NOT include any "location" field.
DO NOT generate geographic text like city or state as a location filter.
Only use:
- country_pref
- state (US only, explicit only)

Rules:

ONLY JSON output (no text)
manufacturer must be UPPERCASE
state must be valid US state codes (e.g., CA, TX, UT)
country_pref must be either "us" or "foreign"

Number and range rules:
"before 2019" → year__lte = 2018
"after 2015" → year__gte = 2016
"1 million" → 1000000

Text normalization rules:
Treat user input as noisy and possibly misspelled
Always correct spelling before extracting filters
Fix aircraft names and common typos
Treat any variation of USA (USA, US, U.S.A, USAd, usd when used as country) as "us"

State filter rules:
Only include "state" if user explicitly mentions specific states
If user says "USA", "US", or "all US", DO NOT include any state filters
If country_pref is "us", state must be empty []

Category IDs:
Jet Aircraft = 1
Turboprop Aircraft = 2
Piston Single Aircraft = 3
Piston Twin Aircraft = 4
Piston Helicopters = 13
Turbine Helicopters = 14

You MUST only process queries related to:
- airplanes
- aircraft
- jets
- helicopters
- aviation
- aircraft manufacturers

You must determine if the user query is a SEARCH query or NOT.
A valid SEARCH query:
- asks to show, list, find, or filter aircraft
- implies browsing inventory
Examples:
- "show jets under 1 million"
- "planes in texas"
- "list cessna aircraft"

An INVALID query:
- asks for definition, explanation, or general knowledge
- is not about filtering or listing aircraft

Examples:
- "how do planes fly"
- "who invented aircraft"
- "hello"
- "random text"

If the user query is NOT related to aircraft/aviation or it is not a SEARCH query,
- Respond with this exact message:
"I only provide information about aircraft, airplanes, jets, helicopters, and aviation-related searches."

Do NOT extract filters.
Do NOT guess intent.
Do NOT respond to unrelated topics.

User query:
{query}

---

You are an aircraft search assistant.

Steps:
1. Extract filters from the user query
2. Call build_url tool
3. Use the URL and call scrape_aircrafts_tool
4. Respond in this format:

If the scraped aircraft list is empty or [], respond exactly:

"No results found."

else respond:
Here are the relevant aircraft I found based on your search:

<list>

You can view all results here:
<url>


IMPORTANT:
- Always call BOTH tools
- Always include URL
- Keep response clean
"""

agent = create_agent(
    model=model,
    tools=[extract_json,build_url, scrape_aircrafts],
    system_prompt=system_prompt
)


In [7]:
def agent_chatbot(query):
    response = agent.invoke({"messages": [("user", query)]})
    return response["messages"][-1].content

user_input = input("Apply filters for aircraft search: ")
print(agent_chatbot(user_input))

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kky4we4afmkarnfpr2n5jf79` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99815, Requested 1208. Please try again in 14m43.872s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}